# Preprocessing

Prepare the data for machine learning by separating the features, target and metadata and handling missing values.

In [16]:
import pandas as pd
import json

training_df = pd.read_csv("../data/processed/training_data.csv")

with open("../data/processed/selected_features.json", "r") as file:
    selected_features = json.load(file)

training_df['date'] = pd.to_datetime(training_df['date'])
training_df = training_df.sort_values(by='date')

# features that the model uses to learn to predict
X = training_df[selected_features]

# label for the correct answer that the model learns to predict
y = training_df['failure_within_30_days']

metadata = training_df[['serial_number', 'date', 'model']]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Metadata shape:", metadata.shape)

X shape: (102123, 27)
y shape: (102123,)
Metadata shape: (102123, 3)


## Missing Values

Check how many missing values are in the selected S.M.A.R.T. features before preprocessing.

In [17]:
print(X.isna().sum())

smart_1_normalized       549
smart_1_raw              549
smart_3_normalized      1053
smart_3_raw             1053
smart_4_normalized      1053
smart_4_raw             1053
smart_5_normalized       828
smart_5_raw              828
smart_7_normalized      1053
smart_7_raw             1053
smart_9_normalized       496
smart_9_raw              496
smart_10_normalized     2048
smart_12_normalized      496
smart_12_raw             496
smart_192_normalized     721
smart_192_raw            721
smart_193_normalized    1063
smart_193_raw           1063
smart_194_normalized     496
smart_194_raw            496
smart_197_normalized    3241
smart_197_raw           3241
smart_198_normalized     882
smart_198_raw            882
smart_199_normalized     829
smart_199_raw            829
dtype: int64


## Missing Value Handling

The remaining missing S.M.A.R.T. values will be handled using median imputation. A missing-value indicator will also be kept so that the model can know when a value was originally missing.

In [18]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median', add_indicator=True)

## Preprocessing Setup

The dataset was separated into 27 selected S.M.A.R.T. features (`X`), the `failure_within_30_days` target (`y`), and metadata. Some of the selected features still have missing values. Median imputation will be used to fill these values and the missing-value indicators will keep track of which values were originally missing. The imputer has not been fitted yet because the data needs to be split first. This prevents information from the validation and test data from being used during preprocessing.

## Time-Aware Split

The data is sorted by the date so the earlier observations can be used for training and later observations can be kept for validation and testing.

In [19]:
training_df = training_df.reset_index(drop=True)

X = training_df[selected_features]
y = training_df['failure_within_30_days']
metadata = training_df[['serial_number', 'date', 'model']]

daily_class_counts = (training_df.groupby('date')['failure_within_30_days'].value_counts())
daily_class_counts = daily_class_counts.unstack(fill_value=0)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Metadata shape:", metadata.shape)
print("Earliest date:", training_df["date"].min())
print("Latest date:", training_df["date"].max())

pd.set_option("display.max_rows", None)
print(daily_class_counts)

X shape: (102123, 27)
y shape: (102123,)
Metadata shape: (102123, 3)
Earliest date: 2026-01-01 00:00:00
Latest date: 2026-03-30 00:00:00
failure_within_30_days     0    1
date                             
2026-01-01              1655  264
2026-01-02              1648  262
2026-01-03              1641  265
2026-01-04              1631  272
2026-01-05              1626  273
2026-01-06              1615  272
2026-01-07              1609  272
2026-01-08              1581  287
2026-01-09              1567  288
2026-01-10              1557  291
2026-01-11              1537  301
2026-01-12              1528  307
2026-01-13              1521  304
2026-01-14              1499  320
2026-01-15              1478  328
2026-01-16              1467  322
2026-01-17              1457  323
2026-01-18              1449  320
2026-01-19              1431  329
2026-01-20              1423  322
2026-01-21              1420  326
2026-01-22              1415  311
2026-01-23              1414  303
2026-01-24   

In [20]:
usable_df = training_df[training_df["date"] <= pd.Timestamp("2026-03-01")].copy()

print("Earliest usable date:", usable_df["date"].min())
print("Latest usable date:", usable_df["date"].max())
print("Usable observations:", len(usable_df))

train_end = pd.Timestamp("2026-02-09")
val_end = pd.Timestamp("2026-02-19")

train_mask = usable_df["date"] <= train_end
val_mask = (usable_df["date"] > train_end) & (usable_df["date"] <= val_end)
test_mask = usable_df["date"] > val_end

print(train_mask.sum())
print(val_mask.sum())
print(test_mask.sum())

Earliest usable date: 2026-01-01 00:00:00
Latest usable date: 2026-03-01 00:00:00
Usable observations: 97786
69511
14619
13656


In [21]:
X_train = usable_df.loc[train_mask, selected_features]
X_val = usable_df.loc[val_mask, selected_features]
X_test = usable_df.loc[test_mask, selected_features]

y_train = usable_df.loc[train_mask, "failure_within_30_days"]
y_val = usable_df.loc[val_mask, "failure_within_30_days"]
y_test = usable_df.loc[test_mask, "failure_within_30_days"]

metadata_columns = ["serial_number", "date", "model"]
metadata_train = usable_df.loc[train_mask, metadata_columns]
metadata_val = usable_df.loc[val_mask, metadata_columns]
metadata_test = usable_df.loc[test_mask, metadata_columns]

print("Train:")
print("X:", X_train.shape)
print("y:", y_train.shape)
print("Metadata:", metadata_train.shape)

print("\nValidation:")
print("X:", X_val.shape)
print("y:", y_val.shape)
print("Metadata:", metadata_val.shape)

print("\nTest:")
print("X:", X_test.shape)
print("y:", y_test.shape)
print("Metadata:", metadata_test.shape)

print("\nTrain dates:")
print(metadata_train["date"].min(), "to", metadata_train["date"].max())
print("\nValidation dates:")
print(metadata_val["date"].min(), "to", metadata_val["date"].max())
print("\nTest dates:")
print(metadata_test["date"].min(), "to", metadata_test["date"].max())

print("\nTrain class counts:")
print(y_train.value_counts())
print("\nValidation class counts:")
print(y_val.value_counts())
print("\nTest class counts:")
print(y_test.value_counts())


train_positive_rate = y_train.mean() * 100
val_positive_rate = y_val.mean() * 100
test_positive_rate = y_test.mean() * 100

print("\nPositive rates:")
print("Train:", train_positive_rate)
print("Validation:", val_positive_rate)
print("Test:", test_positive_rate)

Train:
X: (69511, 27)
y: (69511,)
Metadata: (69511, 3)

Validation:
X: (14619, 27)
y: (14619,)
Metadata: (14619, 3)

Test:
X: (13656, 27)
y: (13656,)
Metadata: (13656, 3)

Train dates:
2026-01-01 00:00:00 to 2026-02-09 00:00:00

Validation dates:
2026-02-10 00:00:00 to 2026-02-19 00:00:00

Test dates:
2026-02-20 00:00:00 to 2026-03-01 00:00:00

Train class counts:
failure_within_30_days
0    57524
1    11987
Name: count, dtype: int64

Validation class counts:
failure_within_30_days
0    11365
1     3254
Name: count, dtype: int64

Test class counts:
failure_within_30_days
0    10448
1     3208
Name: count, dtype: int64

Positive rates:
Train: 17.24475262908029
Validation: 22.258704425747315
Test: 23.491505565319272


## Feature Scaling and Pipelines

Some SMART features have very different numerical scales. Standardization changes each feature so that it has approximately a mean of 0 and a standard deviation of 1.

In [22]:
X_train_means = X_train.mean()
X_train_stds = X_train.std(ddof=0)

feature = 'smart_5_raw'

feature_mean = X_train_means[feature]
feature_std = X_train_stds[feature]


print("Mean:", feature_mean)
print("Standard deviation:", feature_std)

x = X_train[feature].dropna().iloc[0]

print("Original value:", x)

# Calculates how many standard deviations this value is away from the mean
z = (x - feature_mean)/feature_std

print(z)

Mean: 582.9460465520488
Standard deviation: 4249.6112476319995
Original value: 0.0
-0.1371763233347244


In [23]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Gives order to run these steps
# Each stepis a tuple that has the name and what to run
scaled_preprocessor = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("scaler", StandardScaler())])

# Learn the imputation and scaling values from the training data only
scaled_preprocessor.fit(X_train)

# Applies everything that was learned such as replacing NaNs with training medians 
X_train_scaled = scaled_preprocessor.transform(X_train)
X_val_scaled = scaled_preprocessor.transform(X_val)
X_test_scaled = scaled_preprocessor.transform(X_test)

print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

Train: (69511, 54)
Validation: (14619, 54)
Test: (13656, 54)


### Conclusion

I learned how feature scaling and preprocessing pipelines are used to prepare data for machine learning models.

Standardization uses the formula: z = (x - mean) / standard deviation

The mean and standard deviation are calculated separately for each feature. Standard deviation measures how spread out the values of a feature are, while the z-score shows how far a specific value is from the mean. For example, a z-score of 2 means that the value is 2 standard deviations above the mean. I used `StandardScaler` to standardize the features and `SimpleImputer` to handle missing values. These steps were combined into a pipeline so that the data is always processed in the correct order. The pipeline is fitted only on `X_train` because `.fit()` learns information such as the medians, means, and standard deviations from the data. `X_val` and `X_test` are only transformed using the values learned from `X_train`. This keeps the same scale across all three datasets and prevents data leakage from future data. 

For example, if a feature has a training mean of 500 and a standard deviation of 100, a future value of 700 would have a z-score of 2. This means that the value is 2 standard deviations above what was normal in the training data, which allows the model to interpret the value using the same scale it learned during training. Fitting X_val and X_test separately would cause them to use their own means and standard deviations, which would change that scale and allow the preprocessing to learn information from future data which it won't have access to in real environments. 

I also created a separate preprocessing pipeline for tree-based models. Tree-based models generally do not require feature scaling, so this pipeline handles missing values without using `StandardScaler`.